In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Pr₂NiO₄ on 5C2

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the layered nickelate Pr₂NiO₄ to constant-wavelength neutron single-crystal data from 5C2 at
LLB, with anisotropic displacements.

It builds the catalogue case `sc-neut-cwl_pr2nio4-5c2` step by step: the same project the command
line fits and the app's Examples open.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='sc-neut-cwl_pr2nio4-5c2')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('sc-neut-cwl_pr2nio4-5c2-pr2nio4', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['pr2nio4']

Data 'sc-neut-cwl_pr2nio4-5c2-pr2nio4' is at data/sc-neut-cwl_pr2nio4-5c2-pr2nio4.edi


In [5]:
project.display.structure('pr2nio4')

View(html=<359,612 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('sc-neut-cwl_pr2nio4-5c2-llb_5c2-data', destination='data')
project.experiments.add_from_data_path(
    name='llb_5c2',
    data_path=data_path,
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['llb_5c2']
experiment.neutron_scattering_length = 'sears1992'
experiment.instrument.name = '5C2'
experiment.instrument.facility = 'LLB'
experiment.instrument.setup_wavelength = 0.8302
experiment.linked_structures.create(structure_id='pr2nio4', scale=4)

Data 'sc-neut-cwl_pr2nio4-5c2-llb_5c2-data' is at data/sc-neut-cwl_pr2nio4-5c2-llb_5c2.hkl


In [7]:
project.display.pattern('llb_5c2')

View(html=<102,395 characters>, height=480)

## Perform the analysis

A single crystal has no background or profile: the stages are the scale, the positions, the
interstitial oxygen's displacement, then the displacement tensors.

### Scale

The scale brings the calculated intensities to the measured ones.

In [8]:
experiment.linked_structures['pr2nio4'].scale.free = True

In [9]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,llb_5c2,linked_structure,pr2nio4,scale,4,0,-inf,inf,


In [10]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_pr2nio4-5c2, 198 pts fitted / 198 loaded, 1 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                       115.85      6415.49
      1      0.003      49.30      1161.93   81.9% ↓
  done · converged · 1 iters · reduced χ² 1161.93 · wR(F²) 49.30 % · 0.00 s


View(html=<83,334 characters>, height=376)

In [11]:
project.display.pattern('llb_5c2')

View(html=<102,841 characters>, height=480)

### Positions

The coordinates each site's symmetry leaves free.

In [12]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [13]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_pr2nio4-5c2, 198 pts fitted / 198 loaded, 6 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        49.30      1192.18
      1      0.005      46.64      1067.00   10.5% ↓
      2      0.006      46.63      1066.25   0.1% ↓
      3      0.007      46.61      1065.55   0.1% ↓
      4      0.008      46.60      1065.20   
      5      0.009      46.60      1064.86   
      6      0.010      46.59      1064.67   
      7      0.011      46.59      1064.48   
      8      0.011      46.59      1064.37   
      9      0.012      46.58      1064.26   
  done · converged · 9 iters · reduced χ² 1064.26 · wR(F²) 46.58 % · 0.01 s


View(html=<90,639 characters>, height=376)

In [14]:
project.display.pattern('llb_5c2')

View(html=<103,313 characters>, height=480)

### The interstitial displacement

The isotropic displacement of the interstitial oxygen Od.

In [15]:
structure.atom_sites['Od'].adp_iso.free = True

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_pr2nio4-5c2, 198 pts fitted / 198 loaded, 7 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        46.58      1069.84
      1      0.005      46.52      1066.80   0.3% ↓
  done · converged · 1 iters · reduced χ² 1066.80 · wR(F²) 46.52 % · 0.01 s


View(html=<91,863 characters>, height=376)

In [17]:
project.display.pattern('llb_5c2')

View(html=<103,275 characters>, height=480)

### Anisotropic displacements

Last, the tensors, each with the components its site symmetry leaves free.

In [18]:
for tensor in structure.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [19]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_pr2nio4-5c2, 198 pts fitted / 198 loaded, 23 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        46.52      1164.34
      1      0.008       4.29         9.88   99.2% ↓
      2      0.009       4.27         9.79   0.9% ↓
      3      0.011       4.26         9.76   0.3% ↓
      4      0.012       4.26         9.75   0.1% ↓
      5      0.014       4.26         9.75   
      6      0.015       4.26         9.75   
  done · converged · 6 iters · reduced χ² 9.75 · wR(F²) 4.26 % · 0.02 s


View(html=<109,314 characters>, height=376)

In [20]:
project.display.pattern('llb_5c2')

View(html=<103,333 characters>, height=480)

In [21]:
project.display.fit.correlations()

,1,2,3,4,5
1 pr2nio4.atom_site.Od.fract_x,,,,,
2 pr2nio4.atom_site_aniso.Ni.adp_11,0.16,,,,
3 pr2nio4.atom_site_aniso.Ni.adp_22,0.07,0.13,,,
4 pr2nio4.atom_site_aniso.O2.adp_22,0.55,0.18,0.15,,
5 llb_5c2.linked_structure.pr2nio4.scale,0.15,0.56,0.55,0.21,


In [22]:
project.display.structure('pr2nio4')

View(html=<367,204 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [23]:
project.report.save_html('sc-neut-cwl_pr2nio4-5c2.html')

Saving report to 'sc-neut-cwl_pr2nio4-5c2.html'


## Save the project

In [24]:
project.save_as('sc-neut-cwl_pr2nio4-5c2')

Saving project to 'sc-neut-cwl_pr2nio4-5c2'
